In [0]:
import dlt
from pysaprk.sql.functions import *

@dlt.table(
  name="vehicle_bronze",
  comment="Raw vehicle telemetry data from ADLS"
)

def vehicle_bronze_layer():
    return (spark.read.fomat("csv").option("header",True).load("/Volumes/databricks_practice/inputdb/vehicle/gcs/vehicle_telemetry.csv"))

@dlt.table(
  name="vehicle_silver",
  comment="Cleaned and enriched vehicle data"
)
@dlt.expect("valid_fuel_level", "fuel_level BETWEEN 0 AND 100")
@dlt.expect("valid_engine_temp", "engine_temp > 0")

def vehicle_silver():
    df=dlt.read('vehicle_bronze')
    return (df.withColum( "fuel_status",when(col("fuel_level")>15,"LOW FUEL").otherwise('NORMAL')).withColumn(
            "service_due",
            when(
                (col("odometer") - col("last_service_km")) > 10000,
                "YES"
            ).otherwise("NO")
        )
        .withColumn(
            "engine_alert",
            when(col("engine_temp") > 100, "OVERHEAT").otherwise("NORMAL")
        )
    ) 

@dlt.table(
  name="vehicle_gold_daily_summary",
  comment="Daily vehicle monitoring metrics"
)
def vehicle_gold():
    df = dlt.read("vehicle_silver")

    return (
        df
        .groupBy("event_date")
        .agg(
            count("car_id").alias("total_cars"),
            sum(when(col("fuel_status") == "LOW FUEL", 1).otherwise(0)).alias("low_fuel_cars"),
            sum(when(col("service_due") == "YES", 1).otherwise(0)).alias("service_due_cars")
        )
    )


